In [2]:
import pandas as pd
import kagglehub
import os
from pathlib import Path

# 1. Definindo os caminhos das pastas (Arquitetura de Medalhão)
silver_path = Path("Data/Silver")
silver_path.mkdir(parents=True, exist_ok=True)

print("🛰️ Baixando dataset de jogadores do Kaggle...")
# 2. Download do dataset (Sempre pega a versão mais recente)
path_raw = kagglehub.dataset_download("hubertsidorowicz/football-players-stats-2025-2026")
csv_file = os.path.join(path_raw, "players_data_light-2025_2026.csv")

# 3. Lendo e Filtrando
print("🇮🇹 Filtrando jogadores da 'it Serie A'...")
df_raw = pd.read_csv(csv_file)
df_serie_a = df_raw[df_raw['Comp'] == 'it Serie A'].copy()

# 4. Criando a coluna de 'Date' (Evitando o erro do sort_values)
# Como o CSV original é um resumo, usamos o Rank (Rk) para simular o tempo
if 'Date' not in df_serie_a.columns:
    df_serie_a['Date'] = pd.to_datetime('2026-01-01') + pd.to_timedelta(df_serie_a['Rk'], unit='D')

# 5. Salvando na Camada Silver (Parquet é vida!)
output_file = silver_path / "serie_a_silver.parquet"
df_serie_a.to_parquet(output_file, index=False)

print(f"✨ Sucesso! O df_serie_a foi criado com {len(df_serie_a)} jogadores.")
print(f"📁 Salvo em: {output_file}")

# Exibindo os primeiros registros para conferir
df_serie_a.head()

c:\Users\kaiki\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🛰️ Baixando dataset de jogadores do Kaggle...
🇮🇹 Filtrando jogadores da 'it Serie A'...
✨ Sucesso! O df_serie_a foi criado com 578 jogadores.
📁 Salvo em: Data\Silver\serie_a_silver.parquet


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Date
12,13,Zakaria Aboukhlal,ma MAR,MF,Torino,it Serie A,26.0,2000.0,15,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-14
18,19,Francesco Acerbi,it ITA,DF,Inter,it Serie A,38.0,1988.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-20
22,23,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,26,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-24
26,27,Jayden Addai,nl NED,MF,Como,it Serie A,20.0,2005.0,12,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28
33,34,Vasilije Adžić,me MNE,MF,Juventus,it Serie A,19.0,2006.0,10,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-04


#2 Dados StatsBomb

In [9]:
# Carregar dados de eventos do StatsBomb (CSV ou JSON)
import glob
import json

path_sb = kagglehub.dataset_download("saurabhshahane/statsbomb-football-data")
file_sb = os.path.join(path_sb, "events.csv")

if os.path.exists(file_sb):
    df_sb = pd.read_csv(file_sb)
    print("\n" + "="*60)
    print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - CSV)")
    print("="*60)
    print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
    print("\n--- AMOSTRA DOS DADOS ---")
    print(df_sb.head())
else:
    # Fallback para o formato real desta base: JSONs em data/events
    event_json_files = glob.glob(os.path.join(path_sb, "**", "events", "*.json"), recursive=True)
    print(f"events.csv não encontrado. JSONs de eventos encontrados: {len(event_json_files)}")

    if not event_json_files:
        print(f"Nenhum arquivo de eventos foi encontrado em {path_sb}")
    else:
        records = []
        max_files = 5  # amostra para carregar rápido
        for fp in event_json_files[:max_files]:
            with open(fp, "r", encoding="utf-8") as f:
                events = json.load(f)
            if isinstance(events, list):
                records.extend(events)

        if records:
            df_sb = pd.json_normalize(records)
            print("\n" + "="*60)
            print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)")
            print("="*60)
            print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
            print("\nPrimeiras colunas:")
            print(df_sb.columns[::].tolist())
            print("\n--- AMOSTRA DOS DADOS ---")
            print(df_sb.head())
        else:
            print("JSONs encontrados, mas nenhum registro válido foi carregado.")

df_serie_a.head()

events.csv não encontrado. JSONs de eventos encontrados: 3464

ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)
Linhas: 18,359 | Colunas: 130

Primeiras colunas:
['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type.id', 'type.name', 'possession_team.id', 'possession_team.name', 'play_pattern.id', 'play_pattern.name', 'team.id', 'team.name', 'tactics.formation', 'tactics.lineup', 'related_events', 'location', 'player.id', 'player.name', 'position.id', 'position.name', 'pass.recipient.id', 'pass.recipient.name', 'pass.length', 'pass.angle', 'pass.height.id', 'pass.height.name', 'pass.end_location', 'pass.body_part.id', 'pass.body_part.name', 'pass.type.id', 'pass.type.name', 'carry.end_location', 'pass.switch', 'pass.outcome.id', 'pass.outcome.name', 'ball_receipt.outcome.id', 'ball_receipt.outcome.name', 'under_pressure', 'duel.type.id', 'duel.type.name', 'pass.aerial_won', 'counterpress', 'interception.outcome.id', 'interception.outcome.name', 'off_

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Date
12,13,Zakaria Aboukhlal,ma MAR,MF,Torino,it Serie A,26.0,2000.0,15,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-14
18,19,Francesco Acerbi,it ITA,DF,Inter,it Serie A,38.0,1988.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-20
22,23,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,26,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-24
26,27,Jayden Addai,nl NED,MF,Como,it Serie A,20.0,2005.0,12,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28
33,34,Vasilije Adžić,me MNE,MF,Juventus,it Serie A,19.0,2006.0,10,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-04


##Jogos barcelonas

In [17]:
import json
import os

# 1. Ajustando o caminho com a pasta 'data' que você encontrou
path_matches = os.path.join(path_sb, "data", "matches", "11", "2.json")

# 2. Carregando o índice de partidas
with open(path_matches, "r", encoding="utf-8") as f:
    matches_data = json.load(f)

# 3. Filtrando apenas os jogos do Barcelona (Temporada 15/16)
barca_matches = [
    m['match_id'] for m in matches_data 
    if m['home_team']['home_team_name'] == 'Barcelona' 
    or m['away_team']['away_team_name'] == 'Barcelona'
]

print(f"🏟️ SUCESSO! Encontrei {len(barca_matches)} jogos do Barcelona na temporada 15/16.")

# 4. Criando a lista de arquivos de EVENTOS (também dentro de 'data')
lista_arquivos = [
    os.path.join(path_sb, "data", "events", f"{mid}.json") 
    for mid in barca_matches
]

# Teste de segurança: ver se o primeiro arquivo de evento realmente existe
if os.path.exists(lista_arquivos[0]):
    print(f"✅ Arquivos de eventos localizados! Exemplo: {os.path.basename(lista_arquivos[0])}")
else:
    print("❌ Erro: A pasta 'events' não está onde imaginamos.")

🏟️ SUCESSO! Encontrei 34 jogos do Barcelona na temporada 15/16.
✅ Arquivos de eventos localizados! Exemplo: 266989.json


In [18]:
import polars as pl

# 1. Extraindo apenas o essencial para a conferência humana
resumo_jogos = []
for m in matches_data:
    if m['match_id'] in barca_matches:
        resumo_jogos.append({
            "ID": m['match_id'],
            "Data": m['match_date'],
            "Casa": m['home_team']['home_team_name'],
            "Placar": f"{m['home_score']} x {m['away_score']}",
            "Fora": m['away_team']['away_team_name'],
            "Semana": m['match_week']
        })

# 2. Transformando em DataFrame Polars e ordenando por data
df_verificacao = pl.DataFrame(resumo_jogos).sort("Data")

# 3. Printando a tabela (O Polars deixa ela bonitona no VS Code)
print("📅 CALENDÁRIO DO BARCELONA ENCONTRADO (TEMPORADA 15/16):")
print(df_verificacao)

# 4. Verificação rápida: Tem El Clásico?
el_clasico = df_verificacao.filter(
    (pl.col("Casa").str.contains("Real Madrid")) | 
    (pl.col("Fora").str.contains("Real Madrid"))
)

if not el_clasico.is_empty():
    print("\n⚔️ EL CLÁSICO DETECTADO! O Messi vai ter trabalho aqui:")
    print(el_clasico)

📅 CALENDÁRIO DO BARCELONA ENCONTRADO (TEMPORADA 15/16):
shape: (34, 6)
┌────────┬────────────┬───────────────┬────────┬──────────────────┬────────┐
│ ID     ┆ Data       ┆ Casa          ┆ Placar ┆ Fora             ┆ Semana │
│ ---    ┆ ---        ┆ ---           ┆ ---    ┆ ---              ┆ ---    │
│ i64    ┆ str        ┆ str           ┆ str    ┆ str              ┆ i64    │
╞════════╪════════════╪═══════════════╪════════╪══════════════════╪════════╡
│ 267670 ┆ 2016-08-20 ┆ Barcelona     ┆ 6 x 2  ┆ Real Betis       ┆ 1      │
│ 266892 ┆ 2016-08-28 ┆ Athletic Club ┆ 0 x 1  ┆ Barcelona        ┆ 2      │
│ 266669 ┆ 2016-09-10 ┆ Barcelona     ┆ 1 x 2  ┆ Deportivo Alavés ┆ 3      │
│ 267212 ┆ 2016-09-17 ┆ Leganés       ┆ 1 x 5  ┆ Barcelona        ┆ 4      │
│ 267220 ┆ 2016-09-21 ┆ Barcelona     ┆ 1 x 1  ┆ Atlético Madrid  ┆ 5      │
│ …      ┆ …          ┆ …             ┆ …      ┆ …                ┆ …      │
│ 267101 ┆ 2017-04-26 ┆ Barcelona     ┆ 7 x 1  ┆ Osasuna          ┆ 34     │
│ 267

In [10]:
import os
import json
import pandas as pd

# 1. Lista das ligas que você quer comparar (IDs do StatsBomb)
# Bundesliga (9), La Liga (11), World Cup (43), Champions League (16)
competicoes_interesse = [9, 11, 43, 16]

resumo_dados = []

for _, row in df_comps.iterrows():
    if row['competition_id'] in competicoes_interesse:
        comp_id = row['competition_id']
        season_id = row['season_id']
        
        # Caminho para a pasta de matches desta competição/temporada
        path_matches = os.path.join(path_sb, "data", "matches", str(comp_id), f"{season_id}.json")
        
        if os.path.exists(path_matches):
            with open(path_matches, "r", encoding="utf-8") as f:
                matches = json.load(f)
                num_jogos = len(matches)
                
                resumo_dados.append({
                    'Liga': row['competition_name'],
                    'Temporada': row['season_name'],
                    'Qtd_Jogos': num_jogos,
                    'Comp_ID': comp_id,
                    'Season_ID': season_id
                })

# 2. Criar um ranking de volume
df_ranking = pd.DataFrame(resumo_dados).sort_values(by='Qtd_Jogos', ascending=False)

print("📊 VOLUME DE DADOS POR LIGA (StatsBomb):")
print(df_ranking.to_string(index=False))

📊 VOLUME DE DADOS POR LIGA (StatsBomb):
            Liga Temporada  Qtd_Jogos  Comp_ID  Season_ID
         La Liga 2015/2016        380       11         27
   1. Bundesliga 2015/2016        306        9         27
  FIFA World Cup      2022         64       43        106
  FIFA World Cup      2018         64       43          3
         La Liga 2014/2015         38       11         26
         La Liga 2011/2012         37       11         23
         La Liga 2017/2018         36       11          1
         La Liga 2020/2021         35       11         90
         La Liga 2009/2010         35       11         21
   1. Bundesliga 2023/2024         34        9        281
         La Liga 2018/2019         34       11          4
         La Liga 2016/2017         34       11          2
         La Liga 2019/2020         33       11         42
         La Liga 2010/2011         33       11         22
         La Liga 2012/2013         32       11         24
         La Liga 2013/2014      

In [30]:
import pandas as pd
from datetime import datetime

# ====================== FUNÇÃO ATUALIZADA ======================
def extrair_msn_final_com_gols(matches_data, path_base, ids_barca):
    base_msn = []
    
    for match in [m for m in matches_data if m['match_id'] in ids_barca]:
        m_id = match['match_id']
        data_j = datetime.strptime(match['match_date'], '%Y-%m-%d')
        caminho = os.path.join(path_base, "data", "events", f"{m_id}.json")
        
        with open(caminho, "r", encoding="utf-8") as f:
            eventos = json.load(f)
        
        for ev in eventos:
            nome = ev.get("player", {}).get("name")
            if nome not in NASCIMENTOS:
                continue
                
            tipo = ev.get("type", {}).get("name")
            if not tipo:
                continue
                
            # Desfecho mais preciso
            outcome = None
            if "shot" in ev:
                outcome = ev["shot"].get("outcome", {}).get("name")
            elif "pass" in ev:
                outcome = ev["pass"].get("outcome", {}).get("name")
            elif "duel" in ev:
                outcome = ev["duel"].get("outcome", {}).get("name")
            elif "ball_receipt" in ev:
                outcome = ev["ball_receipt"].get("outcome", {}).get("name")
            else:
                outcome = ev.get("outcome", {}).get("name")
            
            nasc = datetime.strptime(NASCIMENTOS[nome], '%Y-%m-%d')
            idade = (data_j - nasc).days // 365

            base_msn.append({
                "match_id": m_id,
                "match_date": match['match_date'],
                "jogador": "Messi" if "Messi" in nome else "Suárez",
                "idade": idade,
                "minuto": ev.get("minute"),
                "segundo": ev.get("second"),
                "tipo": tipo,
                "sob_pressao": ev.get("under_pressure", False),
                "x": ev.get("location", [None, None])[0],
                "y": ev.get("location", [None, None])[1],
                "xg": ev.get("shot", {}).get("statsbomb_xg"),
                "desfecho": outcome if outcome else "Sem desfecho"
            })
    
    df = pd.DataFrame(base_msn)
    df = df.sort_values(by=["match_id", "minuto", "segundo"]).reset_index(drop=True)
    return df


# ====================== EXECUÇÃO ======================
df_msn = extrair_msn_final_com_gols(matches_data, path_sb, barca_matches)

# ==================== FILTRAR GOLS ====================
gols = df_msn[
    (df_msn['tipo'] == 'Shot') & 
    (df_msn['desfecho'] == 'Goal')
].copy()

print(f"🎯 Total de gols do MSN na temporada 15/16: {len(gols)}")
print(f"Messi: {len(gols[gols['jogador']=='Messi'])} | Suárez: {len(gols[gols['jogador']=='Suárez'])}\n")

# ==================== MOSTRAR 5 EVENTOS ANTERIORES PARA CADA GOL ====================
for idx, gol in gols.iterrows():
    match_id = gol['match_id']
    minuto_gol = gol['minuto']
    jogador = gol['jogador']
    
    # Pega todos os eventos dessa partida
    partida = df_msn[df_msn['match_id'] == match_id].copy()
    
    # Encontra a posição do gol no DataFrame da partida
    pos_gol = partida.index[partida['minuto'] == minuto_gol][0]  # pega a primeira ocorrência
    
    # Pega os 5 eventos antes (se existirem)
    eventos_antes = partida.iloc[max(0, pos_gol-5):pos_gol].copy()
    
    print("="*80)
    print(f"⚽ GOL de {jogador} - Partida {match_id} | {gol['match_date']} | {minuto_gol}' | xG: {gol['xg']:.3f}")
    print(f"Local: ({gol['x']:.1f}, {gol['y']:.1f}) | Idade: {gol['idade']} anos")
    print("-"*60)
    print(eventos_antes[['minuto', 'segundo', 'jogador', 'tipo', 'desfecho', 'sob_pressao', 'x', 'y']].to_string(index=False))
    print("\n")

🎯 Total de gols do MSN na temporada 15/16: 64
Messi: 37 | Suárez: 27

⚽ GOL de Messi - Partida 265952 | 2017-03-01 | 8' | xG: 0.302
Local: (104.3, 37.1) | Idade: 29 anos
------------------------------------------------------------
 minuto  segundo jogador          tipo     desfecho  sob_pressao    x    y
      6       23   Messi         Carry Sem desfecho         True 90.9 37.1
      6       26   Messi    Miscontrol Sem desfecho        False 86.1 27.9
      6       46  Suárez Ball Receipt*   Incomplete        False 91.1 56.9
      7       52   Messi Ball Receipt* Sem desfecho         True 92.8 32.1
      7       52   Messi          Pass Sem desfecho         True 92.8 32.1


⚽ GOL de Suárez - Partida 265952 | 2017-03-01 | 10' | xG: 0.025
Local: (118.1, 29.4) | Idade: 30 anos
------------------------------------------------------------
 minuto  segundo jogador          tipo     desfecho  sob_pressao     x    y
      9       13  Suárez Ball Receipt*   Incomplete        False 106.0 25.2
  

In [27]:
pd.DataFrame(df_msn)

,match_id,jogador,idade,minuto,tipo,sob_pressao,x,y,xg,desfecho
0,266989,Messi,29,0,Ball Receipt*,False,55.2,72.8,NaN,Complete/Success
1,266989,Messi,29,0,Pass,False,55.2,72.8,NaN,Complete/Success
2,266989,Suárez,29,1,Ball Recovery,False,12.9,30.6,NaN,Complete/Success
3,266989,Suárez,29,1,Carry,False,12.9,30.6,NaN,Complete/Success
4,266989,Suárez,29,1,Pass,False,14.6,30.1,NaN,Complete/Success
...,...,...,...,...,...,...,...,...,...,...
11161,267058,Messi,29,88,Shot,False,106.6,32.0,0.166324,Off T
11162,267058,Messi,29,90,Ball Receipt*,True,65.0,44.5,NaN,Complete/Success
11163,267058,Messi,29,90,Carry,True,65.0,44.5,NaN,Complete/Success
11164,267058,Messi,29,90,Foul Won,True,79.7,44.5,NaN,Complete/Success


In [ ]:
# Carregar eventos a partir dos JSONs do StatsBomb
import json

# Arquivos JSON de eventos (geralmente ficam em pasta 'events')
event_json_files = [p for p in json_files if "events" in p.lower() and p.lower().endswith(".json")]

print(f"Arquivos JSON de eventos encontrados: {len(event_json_files)}")
for p in event_json_files[:5]:
    print("-", p)

# Limitar quantidade para análise inicial (rápido)
MAX_FILES = 50
selected_files = event_json_files[:MAX_FILES]

records = []
for fp in selected_files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            events = json.load(f)
        if isinstance(events, list):
            records.extend(events)
    except Exception as e:
        print(f"Falha ao ler {fp}: {e}")

if records:
    df_sb = pd.json_normalize(records)
    print(f"\ndf_sb carregado de JSON com sucesso -> Linhas: {df_sb.shape[0]:,}, Colunas: {df_sb.shape[1]}")
    print("\nColunas principais:")
    print(df_sb.columns[:20].tolist())
else:
    print("Nenhum evento foi carregado dos arquivos JSON selecionados.")

##teste

In [6]:
import json
import pandas as pd
import os
from pathlib import Path

# 1. Localizar o arquivo de Competitions (O Catálogo)
path_competitions = os.path.join(path_sb, "data", "competitions.json")

with open(path_competitions, "r", encoding="utf-8") as f:
    comps = json.load(f)

df_comps = pd.json_normalize(comps)

# 2. Filtrar pela temporada mais recente (ex: 2020/2021 ou a maior que tiver)
# Ordenamos por season_id para pegar o que há de mais novo no dataset
df_recent_comps = df_comps.sort_values(by="season_id", ascending=False)

print("🏆 Temporadas mais recentes encontradas:")
print(df_recent_comps[['competition_name', 'season_name', 'competition_id', 'season_id']].head(5))

# 3. Pegar os IDs da temporada mais recente disponível (ex: La Liga 2020/2021)
target_comp = df_recent_comps.iloc[0]['competition_id']
target_season = df_recent_comps.iloc[0]['season_id']

# 4. Localizar os Matches dessa temporada específica
path_matches = os.path.join(path_sb, "data", "matches", str(target_comp), f"{target_season}.json")

with open(path_matches, "r", encoding="utf-8") as f:
    matches = json.load(f)

# Pegamos os IDs de todos os jogos dessa temporada "Elite"
match_ids = [m['match_id'] for m in matches]
print(f"✅ Encontrados {len(match_ids)} jogos recentes da temporada {df_recent_comps.iloc[0]['season_name']}.")

🏆 Temporadas mais recentes encontradas:
     competition_name season_name  competition_id  season_id
71  UEFA Women's Euro        2025              53        315
21       Copa America        2024             223        282
68          UEFA Euro        2024              55        282
0       1. Bundesliga   2023/2024               9        281
24       Copa del Rey   1977/1978              87        279
✅ Encontrados 31 jogos recentes da temporada 2025.


In [8]:
records = []
MAX_JOGOS = 3 # Ajuste conforme sua RAM permitir

for mid in match_ids[:MAX_JOGOS]:
    file_path = os.path.join(path_sb, "data", "events", f"{mid}.json")
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            events = json.load(f)
            # Injetamos o match_id para controle
            for e in events: e['match_id'] = mid
            records.extend(events)

df_sb_recent = pd.json_normalize(records)
print(f"💎 Silver Layer atualizada com {df_sb_recent.shape[0]:,} eventos da temporada mais recente!")

💎 Silver Layer atualizada com 13,313 eventos da temporada mais recente!
